# 4.6. Generalization in Classification

지금까진 Fashion-MNIST의 훈련 데이터로 softmax regression을 배웠다.

하지만 우리가 학습하는 진짜 목적은 훈련 데이터를 잘 맞추는게 아니라 새로운 데이터도 올바르게 분류하는 게 목적이다.

## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [1]:
%matplotlib inline

import matplotlib.pyplot as plt 
import torch 
from torch import nn 
from torch.utils.data import DataLoader 
from torchvision import datasets, transforms

torch.manual_seed(42) 

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu128


## 1. Training 성능과 Generalization

Training error: 모델이 학습에 사용한 데이터에서 틀린 비율

Generalization error: 모델이 학습하지 않은 새로운 데이터에서 틀릴 확률

우리가 실제로 관심이 있는 건 Generalization error이다. 하지만 모든 데이터를 모델에 넣어 평가할 수는 없다.

그래서 일반적으로 test set을 이용해 generalization error을 추정한다.

## 2. Test Set

Test set은 모델 학습에 사용하지 않은 데이터이다.

모델을 고정하고 test set에서 평가하면 새로운 데이터에서 보일 성능을 추정할 수 있다.

분류 모델의 test error은 이렇게 계산한다.

$$
\frac{1}{n}\sum_{i=1}^{n}\mathbf{1}\left(
f(\mathbf{x}^{(i)}) \neq y^{(i)}\right)
$$

의미는 단순한다.
틀린 데이터 개수 / 전체 test 데이터 개수

Accuracy와 error의 관계는 다음과 같다.
$$
1 - \text{accuracy}
$$

예를 들어서 test accuracy가 0.82라면 1 - 0.82 = 0.18

test error은 18%인 것이다.

In [ ]:
predictions = torch.tensor([1, 0, 2, 1, 0]) # 1개 틀리도록
labels = torch.tensor([1, 2, 2, 1, 0]) 

incorrect = predictions != labels 

test_error = incorrect.float().mean() 
test_accuracy = (predictions == labels).float().mean() 

print("틀렸는가:", incorrect) 
print("test error:", test_error.item())
print("test accuracy:", test_accuracy.item()) 
print("error + accuracy:", (test_error + test_accuracy).item())

틀렸는가: tensor([False,  True, False, False, False])
test error: 0.20000000298023224
test accuracy: 0.800000011920929
error + accuracy: 1.0


## 3. Population Error

우리가 알고 싶은 것은 test set에서만의 error이 아니다.

앞으로 모델이 만나게 될 전체 데이터 환경에서의 평균 error을 알고 싶다.
이것을 population error 또는 true error이라고 한다.

$$
E_{(\mathbf{x},y)\sim P}
\left[
\mathbf{1}
\left(
f(\mathbf{x}) \neq y
\right)
\right]
$$

여기에서 $P(X,Y)$는 실제 세상에서 입력과 정답이 생성되는 데이터 분포이다.

그런데 population error은 직접 계산할 수 없다.

실제 세상의 모든 데이터를 모을 수 없기 때문이다.

그래서 test error로 population error을 추정한다.

test set이 실제 환경을 잘 대표한다면 test error은 population error의 좋은 추정값이 될 수 있다.

## 4. Test Set은 실제 환경을 잘 대표해야 한다.

Test set은 모델이 실제로 사용될 환경과 비슷한 분포에서 만들어져야 한다.

예를 들어서 훈련과 test data가 모두 깨끗한 사진으로 구성되었는데 실제 서비스에서는 어두운 휴대전화 사진이 들어온다면 test accuracy는 높아도 서비스 성능은 낮을 수 있다.

test error가 population error을 잘 추정하려면 다음 조건이 필요하다.

- test data가 학습에 사용되지 않아야 한다.
- test data가 실제 환경을 대표해야 한다.
- test sample들이 편향되게 선택되지 않아야 한다.
- test set이 충분히 커야 한다.

## 5. Test Set이 클수록 추정이 안정적이다.

Test Set은 전체 데이터 환경에서 일부 Sample만 뽑은 것이다.

그래서 어느 Sample이 선택되었는지에 따라서 test accuracy가 달라질 수 있다.

Test Set이 작으면 영향이 크다.

Test set이 커질수록 test error은 population error에 더 안정적으로 가까워진다.

일반적으로 추정 오차의 크기는 다음 속도로 줄어든다고 한다.

$$
\mathcal{O}
\left(
\frac{1}{\sqrt{n}}
\right)
$$

$n$은 test sample 개수이다.
sample 수에 반비례하지 않고 sample 수의 제곱근에 반비례한다.

## 6. 정확도를 2배 높이려면 데이터는 4배 필요하다.

추정 오차가 다음과 같이 비례한다고 하면

$$
\frac{1}{\sqrt{n}}
$$

Test set을 4배로 늘리면
$$
\frac{1}{2\sqrt{n}}
$$

추정 오차는 절반이 된다.

추정 오차를 10배 줄이려면 test sample이 약 100배 필요하고
100배 줄이려면 test sample이 약 10,000배 필요하다.

데이터를 조금 더 모은다고 추정이 같은 비율로 정밀해지는 것이 아니다.

정밀도를 높일수록 필요한 데이터가 급격하게 증가한다.

In [3]:
test_sizes = torch.tensor([
    100.0,
    400.0,
    1600.0,
    6400.0,
    25600.0,
])

relative_error = 1 / torch.sqrt(test_sizes)

for n, error in zip(test_sizes, relative_error):
    print(
        f"test size: {int(n.item()):>5}, "
        f"relative estimation error: {error.item():.4f}"
    )

test size:   100, relative estimation error: 0.1000
test size:   400, relative estimation error: 0.0500
test size:  1600, relative estimation error: 0.0250
test size:  6400, relative estimation error: 0.0125
test size: 25600, relative estimation error: 0.0063


Test sample이 늘어날 때 통계적인 추정 오차가 어느 속도로 감소하는지 보여준다.

Test set을 100 => 400로 4배 늘렸지만 추정 오차는 0.1에서 0.05로 절반만 감소했다.

## 7. Bernoulli 관점에서 분류 Error 보기

분류 모델이 각 데이터에서 틀렸는지는
두 가지 값만 가진다.

맞음 > 0
틀림 > 1

sample의 error는 Bernoulli random variable로 볼 수 있다.

True error rate를 $\epsilon(f)$라고 하면 error 변수의 분산은 다음과 같다.

$$
\epsilon(f)
\left(
1-\epsilon(f)
\right)
$$

이 값은 error rate가 0.5일 때 가장 크다.

따라서 test error 추정값의 표준편차는
최악의 경우 대략 다음보다 크지 않다.

$$
\sqrt{
\frac{0.25}{n}
}
$$

Test sample이 많아질수록
이 값은 작아진다.

## 8. Test Error의 불확실성

Test accuracy가 85%라고 측정되어서 실제 population accuracy가 정확히 85%라는 뜻은 아니다.

Test set도 전체 데이터 중 일부 sample이기 때문에 통계적 오차가 있다.

D2L에서는 보수적인 최악의 상황을 기준으로 약 95% 수준에서 error을 ±0.01 범위로 추정하려면 대략 10,000개의 test sample이 필요할 수 있다고 설명한다.

예를 들어서 test error이 0.15로 측정됬다면 실제 error이 정확히 0.15라고 단정하는 것 보다는 대략적인 오차 범위를 함께 고려해야 한다.

해석 예시 : 실제 error이 약 0.14 ~ 0.16 범위일 수 있다.

정확한 confidence interval 계산은 가정과 통계 방버에 따라 달라진다.

In [ ]:
import math
# 정규근사를 이용한 예시

def approximate_error_margin(
    test_error,
    num_samples,
    z_value=1.96,
):
    standard_error = math.sqrt(
        test_error * (1 - test_error) / num_samples
    )

    margin = z_value * standard_error

    return margin


test_error = 0.15

for num_samples in [100, 1000, 10000]:
    margin = approximate_error_margin(
        test_error,
        num_samples,
    )

# Test sample에 따라 측정 결과가 안정적인지 불안정한지 확인할 수 있다.
    print(
        f"n={num_samples:>5} | "
        f"test error={test_error:.3f} | "
        f"approximate margin=±{margin:.3f}"
    )

n=  100 | test error=0.150 | approximate margin=±0.070
n= 1000 | test error=0.150 | approximate margin=±0.022
n=10000 | test error=0.150 | approximate margin=±0.007


예를 들어서

Model A accuracy = 90.1%
Model B accuracy = 90.3%

0.2%p 차이가 있다고 해서
곧바로 Model B가 더 좋다고 단정할 수는 없다.

Test set 크기와 통계적 불확실성을 함께 봐야 한다.

## 9. Test Set을 반복해서 사용하면 안 되는 이유

Test Set은 최종 평가를 위해 남겨둔 것이다.

하지만 실제 개발에서는 이런 실수가 발생할 수 있다고 한다.

1. Model A를 학습한다.
2. Test accuracy를 확인한다.
3. 결과가 마음에 들지 않아 모델을 수정한다.
4. 다시 같은 test set으로 평가한다.
5. 가장 높은 test score를 가진 모델을 선택한다.

표면적으로는 test set을 학습 코드에 넣지 않았다.

하지만 test 결과를 보고 모델을 수정했기 때문에 test set의 정보가 모델 선택 과정에 들어갔다.

모델이 test set에 간접적으로 맞춰졌다.

이를 test set overfitting, adaptive overfitting이라고 한다.

## 10. Test Set 정보가 어떻게 누출되는가

예를 들어서 다음 과정을 반복한다고 하자.

learning rate 변경
→ test accuracy 확인

layer 수 변경
→ test accuracy 확인

weight decay 변경
→ test accuracy 확인

epoch 수 변경
→ test accuracy 확인

가장 높은 test accuracy 선택

이 경우에 test set은 순수한 최종 평가 데이터가 아니다. Test score가 hyperparameter 선택에 사용되었기 때문이다.

다음 두 방식 모두 test set을 사용한 것이다.

**직접적인 사용**  
Test data로 gradient를 계산하고 모델을 학습한다.

**간접적인 사용**  
Test 결과를 보고 모델 구조나 hyperparameter를 변경한다.

간접적인 사용도 test set 누출이다.

## 11. 여러 모델을 평가할수록 우연히 좋은 모델이 나온다.

성능이 비슷한 모델을 많이 평가하면 그중에 하나는 우연히 test set에서 높은 점수를 받을 수 있다.

예를 들어서 성능이 거의 같은 모델 100개를 만들고 같은 test set으로 모두 평가한다고 했을 때 이중 최고 점수를 받은 모델은 실제 population에 가장 좋은 모델이 아니고 현재 test set에 잘 맞은 모델일 수 있다.

많은 모델 평가
→ 우연히 높은 test score 발견
→ 그 모델을 최종 모델로 선택
→ 실제 환경에서는 기대보다 낮은 성능

그래서 test set은 모델 선택에 반복적으로 쓰는 것보다는 최종 한 번의 평가용으로 따로 남겨두는게 좋다.

## 12. Validation Set이 필요한 이유

Test set을 보호하기 위해서 validation set을 사용한다.

### Training set
모델의 weight와 bias를 학습한다.

### Validation set
다음 항목을 선택한다.

- learning rate
- batch size
- weight decay
- epoch 수
- 모델 구조
- hidden layer 수
- 다른 hyperparameter

### Test set
모든 선택이 끝난 후 최종 모델을 한 번 평가한다.

여러 모델 학습 -> Validation 성능으로 비교 -> 최종 모델 선택 -> test set으로 마지막 평가

Test set을 확인하고 모델을 수정했다면 새로운 Test set이 필요하다.

## 13. Test Set을 가능한 적게 확인해야 한다.

실제 연구나 프로젝트에서 Test set을 정확히 한 번만 확인하기는 어렵다.

원칙은 이렇다.

- 모델 개발에는 validation set을 사용한다.
- test set은 가능한 늦게 확인한다.
- test set 확인 횟수를 최소화한다.
- test 결과를 보고 반복적으로 모델을 변경하지 않는다.
- 중요한 문제일수록 더 엄격하게 관리한다.
- 데이터가 적을수록 test set 오염을 더 경계한다.

Benchmark가 장기간 반복해서 사용되면 많은 연구자가 test set을 본다.

그 경우 연구 커뮤니티가 그 test set에 과적합될 수 있다고 한다.

따라서 오래된 bechmark에서의 작은 성능 향상이 새로운 환경에서도 유지된다고 단정하면 안된다.

## 14. Statistical Learning Theory

지금까지 test set을 이용해 모델의 일반화 성능을 평가했다.

test set은 사후 평가이다.

모델 만듬 -> test set 평가 -> 일반화 했는지 확인

Statistical learning theory는 더 근본적인 것을 다룬다.

- 모델이 왜 일반화할 것으로 기대할 수 있는가?
- 어떤 조건에서 training error와 population error가 가까워지는가?
- 모델 복잡도와 데이터 수는 일반화에 어떤 영향을 주는가? 

Statistical learning theory의 목표 중 하나는 generalization gap을 이론적으로 제한하는 것이다.

## 15. Generalization Gap

Generalization gap은 training error와 population error 사이의 차이다.

예를 들어서

training error = 0.05
population error = 0.15
$$
0.15 - 0.05 = 0.10
$$

Training error는 5%인데 실제 환경에서는 15%를 틀린다는 뜻이다.

Generalization gap이 크다면
모델이 훈련 데이터에 지나치게 맞춰졌을 가능성이 있다.

## 16. 고정된 모델과 학습된 모델은 다르다.

이미 고정된 모델이 있다고 했을때

이 모델을 새로운 test set으로 평가하면 test error은 population error의 추정값이 된다.

training set으로 모델을 학습한 뒤 같은 training set에서 error를 측정하면 문제가 달라진다. 모델이 training set의 특징에 맞춰 선택되었기 때문이다.

### 고정된 모델
모델을 먼저 고정
→ 새로운 데이터로 평가

비교적 단순한 평균 추정 문제다.

### 학습된 모델
training data를 보고
가장 잘 맞는 모델을 선택
→ 같은 training data로 평가

모델 선택 자체가 training data에 의존한다.

따라서 training error가 population error보다
지나치게 낮게 나올 수 있다.

## 17. Model Class

모델 하나만 고려하는 것이 아니라 선택 가능한 모델들의 집합을 model class라고 한다.

예를 들어서 2차원 데이터에서 다음이 model class가 될 수 있다.
- 가능한 모든 직선 분류기
- 가능한 모든 결정 트리
- 특정 구조를 가진 모든 신경망

수식으로는 model class를 보통 $\mathcal{F}$로 나타낸다.

$$
f \in \mathcal{F}
$$

모델을 학습한다는 것은
model class 안에서 training data를 잘 맞히는
특정 모델 $f$를 선택하는 과정으로 볼 수 있다.

Model class가 유연할수록
더 다양한 패턴을 표현할 수 있다.

하지만 training data의 noise까지 맞출 위험도 증가한다.

## 18. Model Flexibility와 Overfitting

모델이 너무 단순하면
실제 패턴도 충분히 표현하지 못한다.

이를 underfitting이라고 한다.

모델이 지나치게 유연하면
training data의 세부적인 noise까지 맞출 수 있다.

이를 overfitting이라고 한다.

**너무 단순한 모델**
→ training error 높음
→ underfitting 가능성

**적절한 복잡도의 모델**
→ training error 낮음
→ validation error도 낮음

**지나치게 유연한 모델**
→ training error 매우 낮음
→ validation error가 커질 수 있음
→ overfitting 가능성

하지만 모델 복잡도가 크다고
항상 overfitting이 발생하는 것은 아니다.

데이터 수, optimization, regularization, 모델 구조 등
여러 요소가 함께 영향을 준다.

## 19. Uniform Convergence

Statistical learning theory에서는 model class 안의 모든 모델에 대해 training error와 population error가 동시에 가까워지는지 분석한다.

이를 uniform convergence라고 한다.

단순히 선택된 모델 하나만 보는 것이 아니라 선택 가능한 전체 모델 집합을 고려한다.

목표는 대략 다음과 같은 보장을 만드는 것이다.

높은 확률로,
model class 안의 어떤 모델을 선택하더라도
empirical error와 population error의 차이가
일정한 범위를 넘지 않는다.

하지만 model class가 지나치게 크고 유연하면 이런 보장을 만들기 어려워진다.

Training data를 완전히 외울 수 있는 모델 집합은
training error가 0이어도 실제로 일반화하지 않을 수 있다.

## 20. VC Dimension

VC dimension은 model class의 복잡도 또는 유연성을 측정하는 한 가지 이론적 방법이다.

이 model class는 몇 개의 data point까지 임의의 정답 배치를 모두 표현할 수 있는가? 같은 질문과 관련된다.

VC dimension이 높다는 것은 model class가 더 다양한 분류 경계를 표현할 수 있다는 뜻이다.

예를 들어 d차원 입력의 선형 분류 모델은  
일반적으로 VC dimension이 $d+1$이다.

2차원 입력의 직선 분류기는 VC dimension이 3이다.

## 21. 데이터 수, 모델 복잡도, Generalization Gap

D2L에서 제시하는 VC dimension 기반 bound의 핵심 형태는 다음과 같다.

$$
\text{generalization gap}
\lesssim\sqrt{\frac{\text{model complexity}
+
\text{confidence term}}{n}}
$$

데이터 수 $n$이 증가하면 Generalization gap의 이론적 bound가 작아진다.

더 많은 데이터는 일반화에 유리하다.
모델 복잡도가 증가하면 Generalization gap의 bound가 커질 수 있다.

더 유연한 모델   
→ training data를 잘 맞출 수 있음  
→ overfitting 위험도 증가 

더 강한 확률 보장을 원하면  
더 많은 데이터가 필요하다.

일반화는 다음 요소의 균형 문제다.

- 데이터 수
- 모델 복잡도
- 원하는 신뢰 수준

## 22. 이론적 Bound의 한계

Statistical learning theory는
일반화를 이해하는 중요한 수학적 기반을 제공한다.

하지만 전통적인 일반화 bound는
현대 deep neural network에 대해 지나치게 비관적일 수 있다.

이론적 bound만 보면
일반화를 보장하기 위해 매우 많은 데이터가 필요할 수 있다.

하지만 실제 deep learning 모델은
그보다 훨씬 적은 데이터로도 잘 일반화하는 경우가 많다.

이론은 매우 많은 데이터가 필요하다고 예측했는데
실제로는 더 적은 데이터에서도 좋은 성능을 보이는 경우가 있음

따라서 이론적 bound는 다음 용도로 이해하는 것이 좋다.

일반화 문제의 기본 구조 이해
데이터 수가 왜 중요한지 이해
모델 복잡도가 왜 중요한지 이해
training error만 믿으면 안 되는 이유 이해

현재 단계에서 VC dimension 수식을
계산 문제처럼 외울 필요는 없다.

## 23. 오늘의 정리

- 모델의 목적은 training data 암기가 아니라 새로운 데이터에 대한 일반화다.
- Training accuracy가 높아도 test accuracy가 낮을 수 있다.
- Test error는 test set에서 틀린 데이터의 비율이다.
- Accuracy와 error의 합은 1이다.
- Population error는 실제 전체 환경에서의 평균 error다.
- Population error는 직접 관측할 수 없기 때문에 test error로 추정한다.
- Test set은 실제 사용 환경을 대표해야 한다.
- Test set이 클수록 성능 추정이 안정된다.
- 추정 오차는 대략 $1/\sqrt{n}$ 속도로 감소한다.
- 추정을 2배 정밀하게 만들려면 test data가 약 4배 필요하다.
- Test score에도 통계적인 불확실성이 존재한다.
- 작은 성능 차이가 항상 실제 차이를 의미하는 것은 아니다.
- 같은 test set을 반복해서 보면 test set에 간접적으로 overfitting될 수 있다.
- Hyperparameter와 모델 선택에는 validation set을 사용해야 한다.
- Test set은 최종 모델을 평가할 때 가능한 적게 사용해야 한다.
- Generalization gap은 population error와 training error의 차이다.
- Model class가 유연할수록 training data를 잘 맞출 수 있지만 overfitting 위험도 커질 수 있다.
- VC dimension은 model class의 복잡도를 측정하는 한 가지 이론적 방법이다.
- 데이터가 많을수록 generalization에 유리하다.
- 전통적인 이론적 bound는 deep learning의 실제 일반화를 지나치게 비관적으로 예측할 수 있다.
- 현재 단계에서는 복잡한 증명보다 training, validation, test의 역할을 명확히 구분하는 것이 더 중요하다.